# Notebook 02: Pandas for Machine Learning

**Module:** 01 Python Revision  
**Duration:** ~2 hours

---

## Learning Objectives

1. Load and inspect CSV datasets with Pandas
2. Select, filter, and transform DataFrames
3. Handle missing values and encode categoricals
4. Perform groupby aggregations and merges
5. Reproduce data loading from your legacy Day scripts (Colab-free)

## 1. Intuition: Why Pandas?

NumPy handles **homogeneous numerical arrays**. Real datasets have:
- Column names (`price`, `area`, `survived`)
- Mixed types (float, int, string, datetime)
- Missing values (NaN)
- Row indices

Pandas `DataFrame` = labeled table = what every ML dataset looks like before it becomes a NumPy matrix.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path to repo root CSV files (two levels up from this notebook)
REPO_ROOT = Path('../../').resolve()
print('Repo root:', REPO_ROOT)

## 2. Loading Your Datasets

Your Machine-Learning repo includes CSV files used in the original Day tutorials.

In [ ]:
# Load house price dataset (used in Day - 11)
house = pd.read_csv(REPO_ROOT / 'houseprice.csv')
print('=== House Price Dataset ===')
print(f'Shape: {house.shape}')
print(house.head())
print('\nColumns:', house.columns.tolist())
print('\nInfo:')
house.info()

In [ ]:
# Load Titanic dataset (used in Day - 6)
titanic = pd.read_csv(REPO_ROOT / 'TitanicSurvival.csv')
print('=== Titanic Dataset ===')
print(titanic.head())
print(f'\nShape: {titanic.shape}')
print(f'\nSurvival rate: {titanic["Survived"].mean():.2%}')

## 3. Selection: loc vs iloc

- **`loc`** — label-based: `df.loc[row_label, col_name]`
- **`iloc`** — integer-based: `df.iloc[0, 1]`

For ML, you typically extract feature matrix X and target y:

In [ ]:
# Feature-target split (Day - 11 pattern, modernized)
# Original: x = dataset.drop('price', axis='columns'); y = dataset.price

X = house.drop('price', axis='columns')
y = house['price']

print('Features X:')
print(X.head())
print(f'\nTarget y shape: {y.shape}')
print(y.head())

## 4. Exploratory Data Analysis (EDA)

Before any ML model, always explore:
1. Shape, dtypes, missing values
2. Distributions of features and target
3. Correlations
4. Outliers

In [ ]:
print('=== EDA: House Prices ===')
print(house.describe())
print('\nMissing values:')
print(house.isnull().sum())
print('\nCorrelation with price:')
print(house.corr(numeric_only=True)['price'].sort_values(ascending=False))

## 5. Handling Missing Values

Strategies:
- **Drop** rows/columns with too many NaNs
- **Impute** with mean, median, mode
- **Model-based** imputation (Module 03)

Rule: fit imputation on **training set only**, apply to test set (prevent data leakage).

In [ ]:
# Example with heart disease dataset
heart = pd.read_csv(REPO_ROOT / 'heart_Disease.csv')
print('Heart disease shape:', heart.shape)
print('\nMissing values per column:')
missing = heart.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

# Imputation example (demonstration only — proper split in Module 03)
heart_filled = heart.fillna(heart.median(numeric_only=True))
print('\nAfter median imputation, missing:', heart_filled.isnull().sum().sum())

## 6. Encoding Categorical Variables

ML models need numbers, not strings.

| Method | When to use |
|--------|-------------|
| Label encoding | Ordinal categories (low < medium < high) |
| One-hot encoding | Nominal categories (red, blue, green) |

Module 03 covers this in depth with sklearn `ColumnTransformer`.

In [ ]:
# Titanic: encode Sex column
print('Original Sex values:', titanic['Sex'].unique())

# Manual label encoding
sex_map = {'male': 0, 'female': 1}
titanic_encoded = titanic.copy()
titanic_encoded['Sex_encoded'] = titanic_encoded['Sex'].map(sex_map)

# One-hot encoding with pandas
embarked_dummies = pd.get_dummies(titanic['Embarked'], prefix='embarked')
print(embarked_dummies.head())

## 7. Groupby Aggregations

Essential for understanding data before modeling.

In [ ]:
# Survival rate by passenger class
survival_by_class = titanic.groupby('Pclass')['Survived'].agg(['mean', 'count'])
survival_by_class.columns = ['survival_rate', 'count']
print('Survival by class:')
print(survival_by_class)

# Survival by sex and class
print('\nSurvival by sex and class:')
print(titanic.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack())

## 8. Train/Test Split Pattern

Module 03 uses sklearn's `train_test_split`. Here is the manual concept:

In [ ]:
def manual_train_test_split(X, y, test_ratio=0.2, seed=42):
    """Split data into train and test sets."""
    rng = np.random.default_rng(seed)
    n = len(X)
    indices = rng.permutation(n)
    test_size = int(n * test_ratio)
    test_idx = indices[:test_size]
    train_idx = indices[test_size:]
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

X_train, X_test, y_train, y_test = manual_train_test_split(X, y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 9. Reimplementing Day - 11 (Colab-Free)

Your original script used Google Colab file upload. Here is the same workflow locally:

In [ ]:
# Day - 11 reimplemented locally
from sklearn.linear_model import LinearRegression

# Load (already done above)
dataset = pd.read_csv(REPO_ROOT / 'houseprice.csv')

# Visualize
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(dataset['area'], dataset['price'], color='red', marker='*')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price')
plt.title('House Price vs Area (Day - 11 dataset)')
plt.show()

# Train
x = dataset.drop('price', axis='columns')
y = dataset['price']
model = LinearRegression()
model.fit(x, y)

# Predict for 4685 sq ft (same as Day - 11)
pred = model.predict([[4685]])
print(f'Predicted price for 4685 sq ft: {pred[0]:,.2f}')
print(f'Coefficient (slope): {model.coef_[0]:.2f}')
print(f'Intercept: {model.intercept_:,.2f}')

## 10. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Chained indexing `df[col][row]` | Unpredictable with `.loc` | Use `df.loc[row, col]` |
| SettingWithCopyWarning | Modifying a view | Use `.copy()` explicitly |
| Leaking test data into EDA | Overoptimistic metrics | Split first, then explore train only |
| Not checking dtypes | Model errors on strings | Run `.info()` and `.dtypes` |
| Ignoring missing values | NaN propagates silently | Always check `.isnull().sum()` |

## Exercise 3: Titanic EDA

Using `TitanicSurvival.csv`, compute and print:
1. Overall survival rate
2. Survival rate by passenger class
3. Average age of survivors vs non-survivors
4. Count of missing values per column

In [ ]:
# YOUR CODE HERE


## Exercise 4: Feature Matrix

From the Titanic dataset, create:
- X: columns `Pclass`, `Sex_encoded`, `Age`, `Fare` (drop rows with NaN Age)
- y: `Survived`

Print shapes of X and y.

In [ ]:
# YOUR CODE HERE


## Mini Project: EDA Report

Create a complete EDA for `heart_Disease.csv` (this is the Module 01 assignment):
- Summary statistics
- Missing value analysis
- Correlation heatmap
- Distribution plots for key features
- Written observations (3–5 bullet points)

See `exercises/README.md` for full assignment spec.

---

## Summary

- Pandas DataFrame = labeled table for ML datasets
- Always run EDA before modeling
- Split train/test before any preprocessing that learns from data
- Your Day scripts work locally — just replace Colab upload with `pd.read_csv()`

**Next:** [03_Matplotlib_and_Visualization.ipynb](03_Matplotlib_and_Visualization.ipynb)